In [ ]:
%run ./01-config

In [ ]:
class HistoryLoader:
    def __init__(self, env):
        conf = Config()
        self.landing_zone = conf.base_dir_data + '/raw'
        self.test_data_dir = conf.base_dir_data + '/test_data'
        self.catalog = env
        self.db_name = conf.db_name

    def load_data_lookup(self):
        print('Loading data look up table.....', end='')
        spark.sql(f'''insert overwrite table {self.catalog}.{self.db_name}.date_lookup
                    select date, week, year, month,  dayofweek, dayofmonth, dayofyear, week_part
                    from json.`{self.test_data_dir}/6-date-lookup.json/` ''')
        print('Done')

    def load_history(self):
        import time
        start = time.time()
        print('\nStarting historical data loading......')
        self.load_data_lookup()
        print(f'Historical data load completed in {round(time.time() - start, 2)} seconds')
              
    def assert_count(self,table_name, expected_count):
        print(f'Validating records count in {table_name}')
        actual_count = spark.read.table(f'{self.catalog}.{self.db_name}.date_lookup').count()
        assert actual_count == expected_count, f'Expected {expected_count:,} records but found {actual_count:,} in {table_name}'
        print(f'Found {actual_count:,}/ Expected {expected_count:,} in {table_name}: Success')

    def validate(self):
        import time
        start = time.time()
        print(f'\n Starting historical data load validation\n')
        self.assert_count('date_lookup', 365)
        print(f"Historical data load validation completed in {round(time.time() - start, 2)} seconds")